In [ ]:
"""
Bethe-Bloch Energy Loss Calculator and Plotter
=============================================

This notebook implements the Bethe-Bloch formula for energy loss of charged particles
in matter with automatic material property lookup.

Formula: -dE/dx = K * z² * (Z/A) * (1/β²) * [ln(2mₑc²β²γ²Tₘₐₓ/I²) - β²]
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import c, m_e, N_A, e, pi
import warnings
warnings.filterwarnings('ignore')

# Try to use mendeleev for automatic element property lookup
try:
    from mendeleev import element
    HAS_MENDELEEV = True
    print("✓ Mendeleev library available for automatic element lookup")
except ImportError:
    HAS_MENDELEEV = False
    print("⚠ Mendeleev library not available. Install with: pip install mendeleev")
    print("  Using built-in material database instead")

print("Bethe-Bloch Calculator Loaded Successfully!")

# Bethe-Bloch Energy Loss Calculator

This notebook implements the **Bethe-Bloch formula** for calculating energy loss of charged particles passing through matter.

## Features

✨ **Automatic Material Lookup**: Select materials using chemical symbols (Si, Pb, Au, etc.) and the system automatically retrieves Z, A, and material properties

🧪 **Extensive Material Database**: Includes elements from the periodic table plus common compounds (Water, Air, Scintillators, etc.)

🎯 **Multiple Particle Types**: Pre-configured for muons, pions, protons, electrons with easy addition of new particles

📊 **Comprehensive Plotting**: Generate comparison plots for different materials and particles

⚡ **Interactive Calculations**: Functions for specific energy loss calculations

## The Bethe-Bloch Formula

$$-\frac{dE}{dx} = K z^2 \frac{Z}{A} \frac{1}{\beta^2} \left[ \ln\left(\frac{2m_e c^2 \beta^2 \gamma^2 T_{max}}{I^2}\right) - \beta^2 \right]$$

Where:
- $K = 4\pi N_A r_e^2 m_e c^2 = 0.307$ MeV⋅cm²/g  
- $z$ = charge of incident particle
- $Z, A$ = atomic number and mass of medium
- $\beta = v/c$, $\gamma = 1/\sqrt{1-\beta^2}$
- $I$ = mean excitation energy
- $T_{max}$ = maximum energy transfer in single collision

---

**📝 Instructions**: Modify the configuration variables at the top, then run all cells to generate plots!"

In [ ]:
# ===================================================================
# CONFIGURATION SECTION - MODIFY THESE VARIABLES AS NEEDED
# ===================================================================

# Select materials to plot (chemical symbols or names)
SELECTED_MATERIALS = [
    'Si',        # Silicon
    'Ge',        # Germanium  
    'Ar',        # Argon
    'Al',        # Aluminum
    'Fe',        # Iron
    'Pb',        # Lead
    'C',         # Carbon (graphite)
    'Cu',        # Copper
]

# You can also use full names if mendeleev is available
SELECTED_MATERIAL_NAMES = [
    'Water',
    'Air', 
    'Scintillator'  # Organic scintillator
]

# Select particles to simulate
PARTICLES = [
    {'name': 'Muon', 'charge': 1, 'mass': 105.66},      # MeV/c²
    {'name': 'Pion', 'charge': 1, 'mass': 139.57},      # MeV/c²  
    {'name': 'Proton', 'charge': 1, 'mass': 938.27},    # MeV/c²
    {'name': 'Electron', 'charge': 1, 'mass': 0.511},   # MeV/c²
]

# Energy range for calculations
ENERGY_MIN = 1.0      # MeV
ENERGY_MAX = 10000.0  # MeV  
NUM_POINTS = 1000

print(f"Selected materials: {SELECTED_MATERIALS + SELECTED_MATERIAL_NAMES}")
print(f"Selected particles: {[p['name'] for p in PARTICLES]}")
print(f"Energy range: {ENERGY_MIN} - {ENERGY_MAX} MeV")

In [ ]:
# ===================================================================
# MATERIAL DATABASE WITH AUTOMATIC LOOKUP
# ===================================================================

class MaterialDatabase:
    """Automatic material property lookup using mendeleev + manual database."""
    
    def __init__(self):
        # Manual database for compounds and special materials
        # Format: {name: (Z_eff, A_eff, I_eV, density_g_cm3)}
        self.compound_database = {
            'Water': (10, 18.015, 75.0, 1.0),
            'Air': (7.22, 14.4, 85.7, 0.001225), 
            'Scintillator': (5.57, 11.9, 64.7, 1.032),  # Organic scintillator
            'PMMA': (6.56, 13.25, 74.0, 1.19),
            'Polyethylene': (5.37, 11.17, 57.4, 0.94),
            'Steel': (26, 55.8, 286.0, 7.87),  # Approximate
        }
        
        # Mean excitation energies for elements (if mendeleev doesn't have them)
        self.mean_excitation_energies = {
            'H': 19.2, 'He': 41.8, 'Li': 40.0, 'Be': 63.7, 'B': 76.0, 'C': 78.0,
            'N': 82.0, 'O': 95.0, 'F': 115.0, 'Ne': 137.0, 'Na': 149.0, 'Mg': 156.0,
            'Al': 166.0, 'Si': 173.0, 'P': 173.0, 'S': 180.0, 'Cl': 174.0, 'Ar': 188.0,
            'K': 190.0, 'Ca': 191.0, 'Sc': 216.0, 'Ti': 233.0, 'V': 245.0, 'Cr': 257.0,
            'Mn': 272.0, 'Fe': 286.0, 'Co': 297.0, 'Ni': 311.0, 'Cu': 322.0, 'Zn': 330.0,
            'Ga': 334.0, 'Ge': 350.0, 'As': 347.0, 'Se': 348.0, 'Br': 343.0, 'Kr': 352.0,
            'Rb': 363.0, 'Sr': 366.0, 'Y': 379.0, 'Zr': 393.0, 'Nb': 417.0, 'Mo': 424.0,
            'Tc': 428.0, 'Ru': 441.0, 'Rh': 449.0, 'Pd': 470.0, 'Ag': 470.0, 'Cd': 469.0,
            'In': 488.0, 'Sn': 488.0, 'Sb': 487.0, 'Te': 485.0, 'I': 491.0, 'Xe': 482.0,
            'Cs': 488.0, 'Ba': 491.0, 'La': 501.0, 'Ce': 523.0, 'Pr': 535.0, 'Nd': 546.0,
            'Pm': 560.0, 'Sm': 574.0, 'Eu': 580.0, 'Gd': 591.0, 'Tb': 614.0, 'Dy': 628.0,
            'Ho': 650.0, 'Er': 658.0, 'Tm': 674.0, 'Yb': 684.0, 'Lu': 694.0, 'Hf': 705.0,
            'Ta': 718.0, 'W': 727.0, 'Re': 736.0, 'Os': 746.0, 'Ir': 757.0, 'Pt': 790.0,
            'Au': 790.0, 'Hg': 800.0, 'Tl': 810.0, 'Pb': 823.0, 'Bi': 823.0, 'Po': 830.0,
            'At': 825.0, 'Rn': 794.0, 'Fr': 827.0, 'Ra': 826.0, 'Ac': 841.0, 'Th': 847.0,
            'Pa': 878.0, 'U': 890.0
        }

    def get_element_properties(self, symbol):
        """Get properties for a chemical element using mendeleev or built-in data."""
        if HAS_MENDELEEV:
            try:
                elem = element(symbol)
                Z = elem.atomic_number
                A = elem.atomic_weight
                
                # Try to get density (solid at STP)
                density = getattr(elem, 'density', None)
                if density is None:
                    # Fallback densities for common elements
                    density_fallback = {
                        'H': 0.0000899, 'He': 0.0001785, 'C': 2.267, 'N': 0.001251,
                        'O': 0.001429, 'Al': 2.702, 'Si': 2.329, 'Ar': 0.0017837,
                        'Fe': 7.874, 'Cu': 8.96, 'Ge': 5.323, 'Pb': 11.342
                    }
                    density = density_fallback.get(symbol, 1.0)  # Default to 1 g/cm³
                
                # Get mean excitation energy
                I = self.mean_excitation_energies.get(symbol, 10.0 * Z)  # Rough approximation
                
                return Z, A, I, density
                
            except Exception as e:
                print(f"Warning: Could not lookup {symbol} with mendeleev: {e}")
                
        # Fallback to manual lookup for common elements
        manual_data = {
            'H': (1, 1.008, 19.2, 0.0000899),
            'He': (2, 4.003, 41.8, 0.0001785),
            'C': (6, 12.011, 78.0, 2.267),
            'Al': (13, 26.982, 166.0, 2.702), 
            'Si': (14, 28.085, 173.0, 2.329),
            'Ar': (18, 39.948, 188.0, 0.0017837),
            'Fe': (26, 55.845, 286.0, 7.874),
            'Cu': (29, 63.546, 322.0, 8.96),
            'Ge': (32, 72.630, 350.0, 5.323),
            'Pb': (82, 207.2, 823.0, 11.342)
        }
        
        if symbol in manual_data:
            return manual_data[symbol]
        else:
            raise ValueError(f"Element {symbol} not found in database")

    def get_material_properties(self, material_name):
        """Get material properties (Z, A, I, density) for any material."""
        # First check if it's a compound/special material
        if material_name in self.compound_database:
            return self.compound_database[material_name]
        
        # Otherwise assume it's an element symbol
        return self.get_element_properties(material_name)
    
    def list_available_materials(self):
        """List all available materials."""
        elements = list(self.mean_excitation_energies.keys())[:10]  # Show first 10
        compounds = list(self.compound_database.keys())
        
        print("Available elements (first 10):", elements + ["... and more"])
        print("Available compounds:", compounds)
        
        if HAS_MENDELEEV:
            print("⚠ Full periodic table available via mendeleev library")

# Initialize the material database
material_db = MaterialDatabase()
material_db.list_available_materials()

In [ ]:
# ===================================================================
# BETHE-BLOCH CALCULATION FUNCTIONS
# ===================================================================

def calculate_bethe_bloch(kinetic_energy, particle, material_name):
    """
    Calculate Bethe-Bloch energy loss -dE/dx.
    
    Parameters:
    -----------
    kinetic_energy : array-like
        Kinetic energy in MeV
    particle : dict
        Particle properties with 'charge', 'mass' (MeV/c²), 'name'
    material_name : str
        Material name or chemical symbol
        
    Returns:
    --------
    dEdx : array
        Energy loss -dE/dx in MeV·cm²/g
    """
    
    # Physical constants
    K = 0.307075  # MeV·cm²/g (4πNₐrₑ²mₑc²)
    m_e = 0.511   # MeV/c² (electron mass)
    
    # Get material properties
    Z, A, I_eV, density = material_db.get_material_properties(material_name)
    I = I_eV * 1e-6  # Convert eV to MeV
    
    # Particle properties
    z = particle['charge']
    mass = particle['mass']
    
    # Convert to arrays
    T = np.array(kinetic_energy, dtype=float)
    
    # Calculate relativistic parameters
    gamma = (T + mass) / mass
    beta_squared = 1.0 - 1.0 / (gamma**2)
    beta = np.sqrt(np.maximum(beta_squared, 0))
    
    # Avoid issues at very low energies
    valid_mask = (beta > 0.01) & (T > 0.1)
    
    # Calculate maximum energy transfer Tₘₐₓ
    numerator = 2 * m_e * beta_squared * (gamma**2)
    denominator = 1 + 2*gamma*m_e/mass + (m_e/mass)**2
    T_max = numerator / denominator
    
    # Bethe-Bloch formula
    factor1 = K * (z**2) * (Z / A) / beta_squared
    
    # Logarithmic term
    log_arg = (2 * m_e * beta_squared * (gamma**2) * T_max) / (I**2)
    log_arg = np.maximum(log_arg, 1e-10)  # Avoid log(0)
    log_term = np.log(log_arg)
    
    # Full formula (without density correction for simplicity)
    dEdx = factor1 * (log_term - beta_squared)
    
    # Set invalid regions to NaN
    dEdx[~valid_mask] = np.nan
    
    return dEdx

def calculate_specific_case(particle_name, material_name, energy_mev):
    """Calculate and print Bethe-Bloch for a specific case."""
    
    # Find particle
    particle = None
    for p in PARTICLES:
        if p['name'].lower() == particle_name.lower():
            particle = p
            break
    
    if particle is None:
        print(f"Particle {particle_name} not found in PARTICLES list")
        return
    
    try:
        dEdx = calculate_bethe_bloch([energy_mev], particle, material_name)
        
        print(f"\n{'='*50}")
        print(f"SPECIFIC CALCULATION")
        print(f"{'='*50}")
        print(f"Particle: {particle['name']} (mass = {particle['mass']} MeV/c²)")
        print(f"Material: {material_name}")
        print(f"Kinetic Energy: {energy_mev} MeV")
        print(f"Energy Loss: {dEdx[0]:.3f} MeV·cm²/g")
        
        # Get material properties for context
        Z, A, I, rho = material_db.get_material_properties(material_name)
        print(f"Material properties:")
        print(f"  Z = {Z}, A = {A:.1f} g/mol")
        print(f"  I = {I:.1f} eV, ρ = {rho:.3f} g/cm³")
        print(f"{'='*50}")
        
    except Exception as e:
        print(f"Error in calculation: {e}")

print("✓ Bethe-Bloch calculation functions loaded")

In [ ]:
# ===================================================================
# PLOTTING FUNCTIONS  
# ===================================================================

def plot_materials_comparison(particle, materials_list, energies):
    """Plot Bethe-Bloch for different materials with fixed particle."""
    
    plt.figure(figsize=(12, 8))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(materials_list)))
    
    for material, color in zip(materials_list, colors):
        try:
            dEdx = calculate_bethe_bloch(energies, particle, material)
            
            # Plot only valid points
            valid_mask = ~np.isnan(dEdx) & (dEdx > 0)
            if np.any(valid_mask):
                plt.loglog(energies[valid_mask], dEdx[valid_mask], 
                          label=material, color=color, linewidth=2.5)
        except Exception as e:
            print(f"Warning: Could not calculate for {material}: {e}")
    
    plt.xlabel('Kinetic Energy (MeV)', fontsize=14)
    plt.ylabel(r'$-dE/dx$ (MeV·cm²/g)', fontsize=14)
    plt.title(f'Bethe-Bloch Energy Loss: {particle["name"]} in Different Materials', fontsize=16)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_particles_comparison(material, particles_list, energies):
    """Plot Bethe-Bloch for different particles in fixed material."""
    
    plt.figure(figsize=(12, 8))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(particles_list)))
    
    for particle, color in zip(particles_list, colors):
        try:
            dEdx = calculate_bethe_bloch(energies, particle, material)
            
            # Plot only valid points
            valid_mask = ~np.isnan(dEdx) & (dEdx > 0)
            if np.any(valid_mask):
                label = f'{particle["name"]} (m={particle["mass"]:.1f} MeV/c²)'
                plt.loglog(energies[valid_mask], dEdx[valid_mask], 
                          label=label, color=color, linewidth=2.5)
        except Exception as e:
            print(f"Warning: Could not calculate for {particle['name']}: {e}")
    
    plt.xlabel('Kinetic Energy (MeV)', fontsize=14)
    plt.ylabel(r'$-dE/dx$ (MeV·cm²/g)', fontsize=14)
    plt.title(f'Bethe-Bloch Energy Loss: Different Particles in {material}', fontsize=16)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_comprehensive_grid(materials_list, particles_list, energies):
    """Create a comprehensive grid plot of all combinations."""
    
    n_materials = len(materials_list)
    n_particles = len(particles_list)
    
    fig, axes = plt.subplots(n_particles, n_materials, 
                            figsize=(4*n_materials, 3*n_particles),
                            sharex=True, sharey=True)
    
    if n_particles == 1:
        axes = axes.reshape(1, -1)
    if n_materials == 1:
        axes = axes.reshape(-1, 1)
    
    for i, particle in enumerate(particles_list):
        for j, material in enumerate(materials_list):
            ax = axes[i, j]
            
            try:
                dEdx = calculate_bethe_bloch(energies, particle, material)
                valid_mask = ~np.isnan(dEdx) & (dEdx > 0)
                
                if np.any(valid_mask):
                    ax.loglog(energies[valid_mask], dEdx[valid_mask], 'b-', linewidth=2)
                
                ax.set_title(f'{particle["name"]} in {material}', fontsize=11)
                ax.grid(True, alpha=0.3)
                
                if i == n_particles - 1:
                    ax.set_xlabel('Energy (MeV)', fontsize=10)
                if j == 0:
                    ax.set_ylabel(r'$-dE/dx$ (MeV·cm²/g)', fontsize=10)
            
            except Exception as e:
                ax.text(0.5, 0.5, f'Error:\n{str(e)[:30]}...', 
                       transform=ax.transAxes, ha='center', va='center')
    
    plt.tight_layout()
    plt.show()

def interactive_plotter():
    """Interactive function to quickly plot specific cases."""
    print("Interactive Bethe-Bloch Plotter")
    print("Available materials:", SELECTED_MATERIALS + SELECTED_MATERIAL_NAMES)
    print("Available particles:", [p['name'] for p in PARTICLES])
    
    material = input("Enter material name: ")
    particle_name = input("Enter particle name: ")
    
    # Find particle
    particle = None
    for p in PARTICLES:
        if p['name'].lower() == particle_name.lower():
            particle = p
            break
    
    if particle is None:
        print(f"Particle {particle_name} not found!")
        return
    
    # Generate energies and plot
    energies = np.logspace(np.log10(ENERGY_MIN), np.log10(ENERGY_MAX), NUM_POINTS)
    
    try:
        dEdx = calculate_bethe_bloch(energies, particle, material)
        valid_mask = ~np.isnan(dEdx) & (dEdx > 0)
        
        plt.figure(figsize=(10, 6))
        plt.loglog(energies[valid_mask], dEdx[valid_mask], 'b-', linewidth=2.5)
        plt.xlabel('Kinetic Energy (MeV)', fontsize=14)
        plt.ylabel(r'$-dE/dx$ (MeV·cm²/g)', fontsize=14)
        plt.title(f'Bethe-Bloch: {particle["name"]} in {material}', fontsize=16)
        plt.grid(True, alpha=0.3)
        plt.show()
        
    except Exception as e:
        print(f"Error: {e}")

print("✓ Plotting functions loaded")

In [ ]:
# ===================================================================
# GENERATE ENERGY ARRAY AND RUN CALCULATIONS
# ===================================================================

# Generate energy array for calculations
energies = np.logspace(np.log10(ENERGY_MIN), np.log10(ENERGY_MAX), NUM_POINTS)

print(f"Generated {NUM_POINTS} energy points from {ENERGY_MIN} to {ENERGY_MAX} MeV")
print(f"Ready to calculate Bethe-Bloch for {len(PARTICLES)} particles and {len(SELECTED_MATERIALS + SELECTED_MATERIAL_NAMES)} materials")

# Combine selected materials
all_materials = SELECTED_MATERIALS + SELECTED_MATERIAL_NAMES

print("\nSelected materials for plotting:")
for i, material in enumerate(all_materials, 1):
    try:
        Z, A, I, rho = material_db.get_material_properties(material)
        print(f"{i:2d}. {material:15s} (Z={Z:3.1f}, A={A:6.2f}, I={I:5.1f} eV)")
    except Exception as e:
        print(f"{i:2d}. {material:15s} - Error: {e}")

In [ ]:
# ===================================================================
# PLOT 1: MATERIALS COMPARISON FOR EACH PARTICLE
# ===================================================================

# Plot Bethe-Bloch for different materials with each particle
for particle in PARTICLES:
    print(f"\nPlotting {particle['name']} in different materials...")
    plot_materials_comparison(particle, all_materials, energies)

In [ ]:
# ===================================================================
# PLOT 2: PARTICLES COMPARISON IN EACH MATERIAL
# ===================================================================

# Plot Bethe-Bloch for different particles in each material
for material in all_materials[:5]:  # Limit to first 5 materials to avoid too many plots
    print(f"\nPlotting different particles in {material}...")
    try:
        plot_particles_comparison(material, PARTICLES, energies)
    except Exception as e:
        print(f"Error plotting {material}: {e}")

In [ ]:
# ===================================================================
# EXAMPLE CALCULATIONS
# ===================================================================

# Example specific calculations
print("EXAMPLE SPECIFIC CALCULATIONS")
print("=" * 60)

# Calculate some interesting cases
example_cases = [
    ("Muon", "Si", 1000.0),
    ("Proton", "Ar", 500.0),
    ("Electron", "Pb", 100.0),
    ("Pion", "Water", 200.0)
]

for particle_name, material, energy in example_cases:
    calculate_specific_case(particle_name, material, energy)

In [ ]:
# ===================================================================
# INTERACTIVE USE AND CUSTOMIZATION
# ===================================================================

print("\n" + "="*80)
print("BETHE-BLOCH CALCULATOR - READY FOR USE!")
print("="*80)

print("\n🎯 HOW TO USE THIS NOTEBOOK:")
print("1. Modify SELECTED_MATERIALS and PARTICLES in the configuration section")
print("2. Run all cells to generate plots")
print("3. Use calculate_specific_case(particle, material, energy) for individual calculations")
print("4. Use interactive_plotter() for on-demand plotting")

print("\n📚 AVAILABLE FUNCTIONS:")
print("• calculate_bethe_bloch(energies, particle, material)")
print("• calculate_specific_case(particle_name, material_name, energy_mev)")
print("• plot_materials_comparison(particle, materials_list, energies)")
print("• plot_particles_comparison(material, particles_list, energies)")
print("• plot_comprehensive_grid(materials_list, particles_list, energies)")
print("• interactive_plotter()")

print("\n🧪 TO ADD NEW MATERIALS:")
print("• For elements: Just use chemical symbol (e.g., 'Au', 'Ti')")
if HAS_MENDELEEV:
    print("• Mendeleev library automatically provides Z, A, and density")
else:
    print("• Install mendeleev: pip install mendeleev")
print("• For compounds: Add to compound_database in MaterialDatabase class")

print("\n🚀 TO ADD NEW PARTICLES:")
print("• Add to PARTICLES list with: {'name': 'ParticleName', 'charge': z, 'mass': mass_MeV}")

print("\n" + "="*80)

# Uncomment the line below to run the interactive plotter
# interactive_plotter()